drop gridcode 

drop id column 

transform lcccode and drop the code


# libraries and env

In [1]:
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path

def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  # Loads .env from project root (works if run from notebook too)
    env_vars = {
        "IMAGE_FOLDER": os.getenv("IMAGE_FOLDER"),
        "EXTRACTEDDATASET_FOLDER": os.getenv("EXTRACTEDDATASET_FOLDER"),
        "DATASETS_FOLDER": os.getenv("DATASETS_FOLDER"),
        "ElevationDataset": os.getenv("ElevationDataset"),
        "LandCoverDataset": os.getenv("LandCoverDataset"),
        "GeoBoundaries": os.getenv("GeoBoundaries"),
        "EXTRACTEDELEVATION_FOLDER": os.getenv("EXTRACTEDELEVATION_FOLDER"),
        "EXTRACTEDLANDCOVER_FOLDER": os.getenv("EXTRACTEDLANDCOVER_FOLDER"),
        "EXTRACTEDGEOBOUNDARIES_FOLDER": os.getenv("EXTRACTEDGEOBOUNDARIES_FOLDER"),
        "CLEANEDDATASET_FOLDER": os.getenv("CLEANEDDATASET_FOLDER"),
        "CLEANEDELEVATION_FOLDER": os.getenv("CLEANEDELEVATION_FOLDER"),
        "CLEANEDLANDCOVER_FOLDER": os.getenv("CLEANEDLANDCOVER_FOLDER")
    }
    return env_vars

folders = load_environment()
image_folder = folders["IMAGE_FOLDER"]
extractedData_folder = folders["EXTRACTEDDATASET_FOLDER"]
datasets_folder = folders["DATASETS_FOLDER"]
landCover_folder = folders["LandCoverDataset"]
elevation_folder = folders["ElevationDataset"]
geoboundaries_folder = folders["GeoBoundaries"]
extracted_elevation_folder = folders["EXTRACTEDELEVATION_FOLDER"]
extracted_landcover_folder = folders["EXTRACTEDLANDCOVER_FOLDER"]
extracted_geo_boundaries_folder = folders["EXTRACTEDGEOBOUNDARIES_FOLDER"]
cleaned_dataset_folder = folders["CLEANEDDATASET_FOLDER"]
cleaned_elevation_folder = folders["CLEANEDELEVATION_FOLDER"]
cleaned_landcover_folder = folders["CLEANEDLANDCOVER_FOLDER"]


# Read file

In [14]:
# reading Land Cover dataset geojspon file
landcover_gdf = gpd.read_file(f"{extracted_landcover_folder}/landcover_combined.geojson")


c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Several features with id = 1 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


# Cleaning

In [16]:
# dropping gridcode column and id column
landcover_gdf = landcover_gdf.drop(columns=["gridcode", "id"])

KeyError: "['gridcode', 'id'] not found in axis"

In [30]:
# match lcccode to high level categories pkl file

# read pkl file
import pickle
with open(f"{extracted_landcover_folder}/lcc_highlevel_mapping.pkl", "rb") as f:
    lcc_highlevel_mapping = pickle.load(f)



In [31]:
# add lcc_highlevel column to landcover_gdf
landcover_gdf["lcc_highlevel"] = landcover_gdf["lcccode"].map(lcc_highlevel_mapping)
landcover_gdf.head()

,area,lcccode,geometry,lcc_highlevel
0,6.228187e+06,7001 // 8001,"POLYGON ((6.41528 37.08696, 6.43103 37.0855, 6...",Water bodies
1,6.242408e+06,7001 // 8001,"POLYGON ((7.18084 37.07917, 7.17998 37.08091, ...",Water bodies
2,1.482995e+06,7001 // 8001,"POLYGON ((7.37137 37.08194, 7.3709 37.08717, 7...",Water bodies
3,4.590841e+08,21497-121340,"POLYGON ((6.12361 36.68472, 6.12361 36.69306, ...",Forests
4,6.371533e+06,7001 // 8001,"POLYGON ((6.26181 37.02361, 6.26193 37.02514, ...",Water bodies


In [32]:
# display rows where lcc_highlevel column is null
l = landcover_gdf[landcover_gdf["lcc_highlevel"].isnull()]
l["lcccode"].unique()


array([], dtype=object)

In [33]:
# drop lcccode column
landcover_gdf = landcover_gdf.drop(columns=["lcccode"])
landcover_gdf.head()

,area,geometry,lcc_highlevel
0,6.228187e+06,"POLYGON ((6.41528 37.08696, 6.43103 37.0855, 6...",Water bodies
1,6.242408e+06,"POLYGON ((7.18084 37.07917, 7.17998 37.08091, ...",Water bodies
2,1.482995e+06,"POLYGON ((7.37137 37.08194, 7.3709 37.08717, 7...",Water bodies
3,4.590841e+08,"POLYGON ((6.12361 36.68472, 6.12361 36.69306, ...",Forests
4,6.371533e+06,"POLYGON ((6.26181 37.02361, 6.26193 37.02514, ...",Water bodies


In [ ]:
#  saving the cleaned landcover_gdf to geojson file
landcover_gdf.to_file(f"{cleaned_landcover_folder}/landcover_cleaned.geojson", driver="GeoJSON")